In [1]:
"""BigAlpha v1 线上推理入口：加载 transformer_model.json 生成每日分数。"""
import os

import dai
import numpy as np
import pandas as pd
import torch

from transformer_train import (
    MODEL_PATH,
    BATCH,
    BigAlphaV1Model,
    pool,
    load_model,
    predict_scores_streaming,
    _resolve_bar1m_table,
)


def main(datasources, start_date, end_date):
    table = _resolve_bar1m_table(datasources)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError(
            f"Missing model file: {MODEL_PATH}. "
            "Upload transformer_model.json with this notebook and transformer_train.py."
        )

    checkpoint = load_model(MODEL_PATH, map_location=device)
    model_cfg = checkpoint["model_cfg"]
    feature_cols = checkpoint["feature_cols"]
    stats = (
        np.asarray(checkpoint["mean"], np.float32),
        np.asarray(checkpoint["std"], np.float32),
    )
    seq_len = int(checkpoint["seq_len"])
    if int(model_cfg["bars_per_day"]) != 240 or seq_len != 1200:
        raise ValueError("transformer_model.json 不是 1分钟/5交易日权重")

    model = BigAlphaV1Model(**model_cfg).to(device)
    model.load_state_dict(checkpoint["state_dict"], strict=True)
    model.eval()

    instruments = pool(start_date, end_date)
    idx_df = predict_scores_streaming(
        model,
        table,
        start_date,
        end_date,
        instruments,
        stats,
        feature_cols,
        device,
        batch_size=BATCH,
        seq_len=seq_len,
    )
    if idx_df.empty:
        raise ValueError("No inference samples were generated")

    stock_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]},
    ).df()
    stock_pool["date"] = pd.to_datetime(stock_pool["date"]).dt.normalize()
    stock_pool["instrument"] = stock_pool["instrument"].astype(str)
    idx_df["date"] = pd.to_datetime(idx_df["date"]).dt.normalize()
    idx_df["instrument"] = idx_df["instrument"].astype(str)

    result = (
        pd.merge(idx_df, stock_pool, on=["date", "instrument"], how="inner")
        .replace([np.inf, -np.inf], np.nan)
        .dropna(subset=["score"])
        .drop_duplicates(["date", "instrument"])[["date", "instrument", "score"]]
        .reset_index(drop=True)
    )
    return result


if __name__ == "__main__":
    from bigmodule import M

    datasources = {"bar1m": "bigalpha_2026_stock_bar1m"}
    start_date = "2024-01-01 00:00:00"
    end_date = "2024-12-31 23:59:59"
    score_data = main(datasources, start_date, end_date)
    print(score_data.head())
    result = M.bigalpha_eval._latest(factor_data=score_data, show=True)
